<a href="https://colab.research.google.com/github/div652/GPT-2/blob/main/gpt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Goal here is to set up GPT2 architecture
- reproduce results
- ability to load their model
- ability to train our own model
- We will reproduce the 124M model.
- smallest was 117M in their paper, was corrected to 124M later. see github.
- Will refer GPT3 as well, since there is not a huge departure in the architecture.
- Will implement in pytorch, but the initial code was in tensorflow.
- Their a torch implementation in huggingface, and you can load the model from here as well.
- https://huggingface.co/openai-community/gpt2


# Sources
Attention is all you need paper : https://arxiv.org/abs/1706.03762
OpenAI GPT-2 Paper : https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
GPT2 Blogpost : https://openai.com/index/better-language-models/

OpenAI GPT-3 Paper :https://arxiv.org/pdf/2005.14165

We begin by setting up the model

# Imports

In [2]:
import os
import math
import time
import inspect
from dataclasses import dataclass # for GPT Config class
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken

In [3]:
#return a torch tensor of prompt to tokens on cpu.
def get_tokens(prompt):
  enc = tiktoken.get_encoding('gpt2')
  tokens = enc.encode(prompt)
  tokens = torch.tensor(tokens, dtype = torch.long)
  return tokens
def get_text(tokens):
  enc = tiktoken.get_encoding('gpt2')
  decoded = enc.decode(tokens)
  return decoded


# Importing Trained Model to check GPT2 weights

In [4]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
HuggingFaceModel = GPT2LMHeadModel.from_pretrained("gpt2")
HuggingFaceStateDict = HuggingFaceModel.state_dict()
for k,v in HuggingFaceStateDict.items():
  print(k,v.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln_1.weight torch.Size([768])
transformer.h.1.ln_1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias 

# Setting up the tiny-shakespeare dataset

In [19]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(get_tokens(text[:1000]).shape, get_tokens(text[:1000]))

--2026-05-05 17:26:37--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.6’

input.txt.6         100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2026-05-05 17:26:37 (154 MB/s) - ‘input.txt.6’ saved [1115394/1115394]

torch.Size([285]) tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
         3285,   502,  2740,    13,   198,   198,  3237,    25,   198,  5248,
          461,    11,  2740,    13,   198,   198,  5962, 22307,    25,   198,
         1639,   389,   477, 12939,  2138,   284,  4656,   621,   284,  1145,
          680,    30,   198,   198,  3237,    25,   198,  4965,  563

Ok, dataset loading works fine.

Compression rate seems 28%

# Implementing our own model

## Hyper Parameter Setup

In [12]:
@dataclass
class GPTConfig:
  block_size: int=1024 # context length
  vocab_size: int=50257 # 50,000 BPE merges, 256 bytes, 1 EOS
  n_layer: int =12
  n_head: int =12
  n_embd: int =768
  device = "cpu"
  if torch.cuda.is_available():
    device = "cuda"
  elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
  print("Using device: %s" % device)

Using device: cuda


## Model Code

In [6]:
class CausalSelfAttention(nn.Module):
# c_attn.weight torch.Size([768, 2304])
# c_attn.bias torch.Size([2304])
# c_proj.weight torch.Size([768, 768])
# c_proj.bias torch.Size([768])
  def __init__(self,config):
    super().__init__()
    assert config.n_embd%config.n_head==0
    self.n_embd = config.n_embd
    self.n_head = config.n_head

  # using bias here because openAI used it, I don't think it is needed.
    self.c_attn = nn.Linear(self.n_embd, 3*self.n_embd)
    self.c_proj = nn.Linear(self.n_embd, self.n_embd)

    # calling it mask,
    self.register_buffer("mask", torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))

  def forward(self,x):
    B,T,C = x.shape
    qkv = self.c_attn(x)
    q,k,v = torch.split(qkv, self.n_embd, dim=2)

    q = q.view(B,T,self.n_head, C//self.n_head).transpose(1,2) # B,n_h,T,n_embd/n_h
    k = k.view(B,T,self.n_head, C//self.n_head).transpose(1,2) # B,n_h,T,n_embd/n
    v = v.view(B,T,self.n_head, C//self.n_head).transpose(1,2) # B,n_h,T,n_embd/n_h

    att = q@k.transpose(-2,-1) # B,n_h, T, T
    att = att*((C//self.n_head)**-0.5) # scaling by head dimesion
    att = att.masked_fill(self.mask[:,:,:T,:T]==0,float('-inf'))
    att = F.softmax(att,dim=-1)
    y = att@v
    y = y.transpose(1,2).contiguous().view(B,T,C)
    y = self.c_proj(y)
    return y




nn.Gelu has two versions, the original version, and he approximate version calculated from tanh.

https://arxiv.org/pdf/1606.08415was deveopled the erf function which is used in torch for GELU implementation , was very slow in torch at that time. So at that time bert and gpt-2 used the tanh version in their architecutre, so we are going to use that.

Why GELU vs. RELU
- activations that ended up in the zero region had no gradient
- so there is always some contribution in the GELU case, there are not dead regions


In [7]:
class FeedForwardLayer(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.c_fc = nn.Linear(config.n_embd, 4*config.n_embd)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4*config.n_embd, config.n_embd)

  def forward(self,x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x


Note: in the original Attention is all you need paper there was this one difference.
- Layer Norms were after Attention block and MLP . They are now before attention and MLP


In [8]:
class Block(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.ln_1 = nn.LayerNorm(config.n_embd)
    self.attn = CausalSelfAttention(config)
    self.ln_2 = nn.LayerNorm(config.n_embd)
    self.mlp = FeedForwardLayer(config)

# each block is just a repeated application of map-reduce.
  def forward(self,x):
    # attention is the reduce
    x = x + self.attn(self.ln_1(x))
    # mlp is the map
    x = x + self.mlp(self.ln_2(x))
    return x

### Note on why we use no_grad while doing copy
1. Avoiding Computation Graph Overhead
By default, if sd[k] or sd_hf[k] are tensors that have requires_grad=True (which is standard for model parameters), any operation performed on them—including .copy_() or .t() (transpose)—could be tracked by the Autograd engine.

Without no_grad: PyTorch might try to build a "graph" of this operation, consuming extra memory and CPU/GPU cycles to keep track of how the data was moved.

With no_grad: You tell PyTorch, "This is just a memory-to-memory copy; don't worry about the history of these tensors."

2. Preventing "Leaf Tensor" Errors
In PyTorch, model parameters are usually leaf tensors. Autograd generally prevents you from performing "in-place" operations on leaf tensors that require gradients because it would break the ability to calculate gradients for them later.
If you tried to use .copy_() on a parameter without no_grad(), PyTorch might throw a runtime error saying:

"RuntimeError: a view of a leaf Variable that requires grad is being used in an in-place operation."

Using the no_grad context manager signals that you are intentionally modifying the values outside of the training loop.

In [9]:
class GPT(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.config = config
    self.transformer = nn.ModuleDict({
        "wte" : nn.Embedding(config.vocab_size, config.n_embd),
        "wpe" : nn.Embedding(config.block_size, config.n_embd),
        "h" : nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
        "ln_f" : nn.LayerNorm(config.n_embd)
    })
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
# from_pretrained is a constructor that returns a GPT object initialised with
# weights form the hugging face model, based on the model_type we provide it with.
  def forward(self,x):
    # x is a batch of tokens
    B,T = x.shape
    assert T<self.config.block_size , f"Cannot forward sequence of length {T}, block size is {self.config.block_size}"
    pos = torch.arange(T, dtype=torch.long, device=x.device) # should be on the same device
    token_embeddings = self.transformer.wte(x) # B,T,n_embd
    position_embeddings = self.transformer.wpe(pos) # T,n_embd
    x = token_embeddings + position_embeddings # torch will broadcast along Batch dimension.
    for block in self.transformer.h:
      x = block(x)
    x = self.transformer.ln_f(x)
    logits = self.lm_head(x) # (B,T,vocab_size)
    return logits

  @classmethod
  def from_pretrained(cls,model_type):
    # Loads pretrained GPT-2 Model weight from hugging face
    assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
    from transformers import GPT2LMHeadModel
    print("Loading Weights from pretrained gpt: %s" % model_type)

    # n_layer, n_head and n_embd are detemrined from model type
    config_args = {
        'gpt2': dict(n_layer=12, n_head=12, n_embd=768),        # 124M params
        'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),# 350M params
        'gpt2-large': dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
        'gpt2-xl': dict(n_layer=48, n_head=25, n_embd=1600),    # 1558M params
    }[model_type]
    config_args['vocab_size'] = 50257 # same for all the models
    config_args['block_size'] = 1024  # same for all the models
    config = GPTConfig(**config_args)

    # our model
    model = GPT(config)
    sd = model.state_dict()
    sd_keys = sd.keys()
    print(sd_keys,sep='\n')
    sd_keys = [k for k in sd_keys if not k.endswith(".attn.mask")] # ignoring the register buffers, these are non trainable parameters

    # hugging face model
    model_hf = GPT2LMHeadModel.from_pretrained(model_type)
    sd_hf = model_hf.state_dict()
    sd_keys_hf = sd_hf.keys()
    print(sd_keys_hf,sep='\n')
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith(".attn.masked_bias")] # ignoring the register buffers, these are non trainable parameters
    sd_keys_hf = [k for k in sd_keys_hf if not k.endswith(".attn.bias")] # ignoring the register buffers, these are non trainable parameters
    transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
    # some layers in the gpt2 implementation need to transposed since they were originally written as a conv1d.
    # while we have implemented it using a vanilla Linear layer
    assert len(sd_keys)==len(sd_keys_hf),f"Model sizes do not match {len(sd_keys)} vs {len(sd_keys_hf)}"
    for k in sd_keys:
      if any(k.endswith(w) for w in transposed):
        assert sd_hf[k].shape[::-1]==sd[k].shape, f"Shapes {sd_hf[k].shape[::-1]} and {sd[k].shape} don't match for key {k}"
        with torch.no_grad():
          sd[k].copy_(sd_hf[k].t())
      else:
        #vanilla copy over the parameters
        assert sd_hf[k].shape==sd[k].shape , f"Shapes {sd_hf[k].shape} and {sd[k].shape} don't match for key {k}"
        with torch.no_grad():
          sd[k].copy_(sd_hf[k])

    return model


In [10]:
# this worked fine. Currently skipping it.
%%script false --no-raise-error
model = GPT.from_pretrained('gpt2')
print("model loaded successfully")

num_return_sequences = 5
max_length = 30
model.eval() # this removes things like dropouts
model.to('cuda')
tokens = get_tokens("Hello, I'm a language model")
tokens = tokens.unsqueeze(0).repeat(num_return_sequences,1) # repeat along the dim1
x = tokens.to('cuda')
torch.manual_seed(42)
torch.cuda.manual_seed(42)
print(x.shape)
while x.size(1) < max_length:
  with torch.no_grad():
    logits = model(x) # B,T,vocab_size
    logits = logits[:,-1,:] # B,vocab_size. Only need the last logits
    probs = F.softmax(logits,dim=-1)
    # we pick only top50, so that hte model doesn't stray away into unkonwn zones. and styas in "good" zones
    topk_probs, topk_indices = torch.topk(probs,50,dim=-1) # this is the temperature sampling essentially. we don't pick the most probable, rather we sample from the distribution
    ix = torch.multinomial(topk_probs,1)#B,1
    xcol = torch.gather(topk_indices,-1,ix)
    x = torch.cat((x,xcol),dim=1)

  # print the generated text
for i in range(num_return_sequences):
  tokens = x[i,:max_length].tolist()
  print(">",get_text(tokens))

Above results are same when we sample from the original Hugging Face model.

There is a difference when we use pipeline, don't know why.

Lets move ahead.

Next step is to train the model on our own.

model loaded successfully
torch.Size([5, 7])
> Hello, I'm a language model Raymond negotiation 09 industries (# debutedpantsUME plottingernelUGnote introduertonhloret Ah balloon inflict Vine Boxingarians chemist
> Hello, I'm a language model channelsAmericansylvania�doors intermediate VR uglyournament paletteoplan Bett Grav Processopathy Dynamics Ah irrigation�aum unnatural Layr
> Hello, I'm a language model playoffs Francois strange hamstring Console socialist racket ideology sixth paletteerousVirgin stun guarant le moon 11 pensionsDescriptionirty existing smiled Vikings
> Hello, I'm a language model Practvs authors Verd poets Presbyterian broom mirroredContactedy WeinSAgravity Walt496 Frankfurt radikg fiscal realiseowan Melania refurb
> Hello, I'm a language model unexpl PureittyCIbeiternautightJapanese sweetsvisual lastsSAdB eldest vari contraction Vis Michel oy foss pigment minor co
